# LoRA fine-tuning on Google Colab (free T4) — fine-tune-lab

Trains a small encoder classifier with a LoRA adapter to replace the Claude
classification call in `claims-auditor` cheaply. The CPU smoke run uses
BERT-tiny; on a T4 you can use a stronger base (e.g. `distilbert-base-uncased`)
for higher accuracy. **Runtime → Change runtime type → T4 GPU.** Synthetic data only.

In [ ]:
!git clone https://github.com/MarcosRos002/fine-tune-lab.git
%cd fine-tune-lab
!pip -q install -e . transformers peft datasets accelerate scikit-learn

In [ ]:
from fine_tune_lab.data.build_dataset import build_dataset
from fine_tune_lab.train.config import TrainConfig
from fine_tune_lab.train.lora import train_lora, build_predictor
from fine_tune_lab.eval.metrics import evaluate
from fine_tune_lab.eval.cost import cost_per_request
from fine_tune_lab.eval.report import render_table

build_dataset(out_dir='data/processed', n=2000, seed=0)

# On a T4, a stronger base lifts accuracy. distilbert attention = q_lin/v_lin.
cfg = TrainConfig(
    base_model='distilbert-base-uncased',
    target_modules=['q_lin', 'v_lin'],
    num_epochs=5, batch_size=32, output_dir='artifacts/lora',
)
train_lora('data/processed', cfg)

predict = build_predictor(cfg.output_dir, base_model=cfg.base_model)
m = evaluate(predict, 'data/processed/test.jsonl')
m['cost_per_1k'] = cost_per_request(n_requests=1000, input_tokens=120000, output_tokens=20000, backend='vllm') * 1000
print('accuracy:', m['accuracy'])
print(render_table({'Fine-tuned LoRA (vLLM)': m}))